In [1]:
import os, numpy as np, pandas as pd
from google.colab import drive
!pip -q install datasets
from datasets import load_dataset
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/"
BASE += "Project2_Stockout"
os.makedirs(BASE, exist_ok=True)
ds = load_dataset(
    "Dingdong-Inc/FreshRetailNet-50K")
tr = ds["train"]
print(len(tr))

Mounted at /content/drive


README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/eval.parquet: reconstructing file:   0%|          |  0.00B / 8.44MB            

data/eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

4500000


In [2]:
keep = ["city_id", "store_id",
        "management_group_id",
        "first_category_id",
        "second_category_id",
        "third_category_id", "product_id",
        "dt", "sale_amount",
        "stock_hour6_22_cnt", "discount",
        "holiday_flag", "activity_flag",
        "precpt", "avg_temperature",
        "avg_humidity", "avg_wind_level"]
raw = tr.select_columns(keep).to_pandas()
df = raw.copy()
print(raw.shape)

(4500000, 17)


In [3]:
key = ["city_id", "store_id",
       "product_id", "dt"]
print(df.dtypes)
print(df.isna().sum())
print(df.duplicated().sum())
print(df.duplicated(key).sum())
print(df.describe().T.round(3))
print(df["sale_amount"].quantile(
    [0.5, 0.99, 0.999, 1]))
print((df["sale_amount"] < 0).sum())
print(df["stock_hour6_22_cnt"]
      .value_counts().sort_index())
for c in ["holiday_flag", "activity_flag"]:
    print(c, df[c].unique())
n = df.groupby(["city_id", "store_id",
                "product_id"]).size()
print(len(n), n.min(), n.max())
print(df.groupby("product_id")
      ["third_category_id"].nunique().max())
print(df.groupby("store_id")
      ["city_id"].nunique().max())

city_id                  int64
store_id                 int64
management_group_id      int64
first_category_id        int64
second_category_id       int64
third_category_id        int64
product_id               int64
dt                      object
sale_amount            float64
stock_hour6_22_cnt       int32
discount               float64
holiday_flag             int32
activity_flag            int32
precpt                 float64
avg_temperature        float64
avg_humidity           float64
avg_wind_level         float64
dtype: object
city_id                0
store_id               0
management_group_id    0
first_category_id      0
second_category_id     0
third_category_id      0
product_id             0
dt                     0
sale_amount            0
stock_hour6_22_cnt     0
discount               0
holiday_flag           0
activity_flag          0
precpt                 0
avg_temperature        0
avg_humidity           0
avg_wind_level         0
dtype: int64
0
0
                 

In [4]:
def chk(b):
    hs = np.array(b["hours_stock_status"],
                  dtype=np.int8)
    hv = np.array(b["hours_sale"],
                  dtype=np.float32)
    return {"so16": hs[:, 6:22].sum(1),
            "hs_min": hs.min(1),
            "hs_max": hs.max(1),
            "hv_sum": hv.sum(1),
            "hv_min": hv.min(1)}
hc = tr.map(chk, batched=True,
            batch_size=50_000,
            remove_columns=tr.column_names)
hc = hc.to_pandas()
ok = hc["so16"].values == \
     df["stock_hour6_22_cnt"].values
print(len(hc), ok.mean())
print(hc["hs_min"].min(), hc["hs_max"].max())
d = hc["hv_sum"].values - \
    df["sale_amount"].values
print(np.abs(d).max())
print((hc["hv_min"] < 0).sum())

Map:   0%|          | 0/4500000 [00:00<?, ? examples/s]

4500000 1.0
0 1
4.577636715907829e-06
0


In [5]:
# Convert dates, derive stock-out columns, flag (not drop) the odd values, shrink the IDs
df["dt"] = pd.to_datetime(df["dt"], format="%Y-%m-%d")
df["so_day"] = df["stock_hour6_22_cnt"] > 0
df["so_share"] = df["stock_hour6_22_cnt"] / 16

q999 = df["sale_amount"].quantile(0.999)
df["sale_outlier"] = df["sale_amount"] > q999                       # kept: metrics do not use sales
df["disc_odd"] = (df["discount"] <= 0) | (df["discount"] > 1)       # kept: metrics do not use discount
df["so_allday"] = df["stock_hour6_22_cnt"] == 16                    # kept: may mean not offered that day
print(df["sale_outlier"].sum(), "sale outliers above", round(q999, 2), "| max", df["sale_amount"].max())
print((df["discount"] <= 0).sum(), "discounts of 0 |", (df["discount"] > 1).sum(), "discounts above 1")
print(df["so_allday"].sum(), "full-day stock-outs")
print(df.loc[df["so_allday"], "sale_amount"].describe().round(3))

ids = ["city_id", "store_id", "management_group_id", "first_category_id",
       "second_category_id", "third_category_id", "product_id",
       "stock_hour6_22_cnt", "holiday_flag", "activity_flag"]
for c in ids:
    df[c] = pd.to_numeric(df[c], downcast="integer")

4472 sale outliers above 20.6 | max 44.9
16039 discounts of 0 | 34 discounts above 1
181791 full-day stock-outs
count    181791.000
mean          0.046
std           0.201
min           0.000
25%           0.000
50%           0.000
75%           0.000
max          15.400
Name: sale_amount, dtype: float64


In [6]:
assert len(df) == 4_500_000, "rows"
assert not df.duplicated(key).any(), "dup"
assert len(n) == 50_000, "series"
assert (n == 90).all(), "days"
assert df["stock_hour6_22_cnt"].between(
    0, 16).all(), "range"
assert df["stock_hour6_22_cnt"].sum() == \
    hc["so16"].sum(), "hourly"
print(round(df["so_day"].mean(), 4))
print(round(df["so_share"].mean(), 4))

0.4427
0.1988


In [7]:
p = f"{BASE}/fresh_clean.parquet"
df.to_parquet(p, index=False)
print(os.listdir(BASE))

['fresh_clean.parquet', 'dash']


In [8]:
# Instructor checks: all-day impact, category hierarchy, parquet size
a = df[~df["so_allday"]]
print(round(a["so_day"].mean(), 4), round(a["so_share"].mean(), 4))
w = df["stock_hour6_22_cnt"]
print(round(w[df["so_allday"]].sum() / w.sum(), 3))
print(df.groupby("third_category_id")["second_category_id"].nunique().max())
print(df.groupby("second_category_id")["first_category_id"].nunique().max())
print(df["first_category_id"].nunique())
print(round(os.path.getsize(p) / 1e6), "MB")

0.4192 0.165
0.203
1
1
32
19 MB


Cleaning log: FreshRetailNet-50K, train split (p2_01_cleaning)

1. Raw kept untouched; worked on a copy. 4,500,000 rows, 17 columns. The two hourly list columns are not
   loaded into pandas; they are checked in batches.
2. Duplicates: 0 duplicate rows and 0 duplicate store-product-days. 50,000 series, each exactly 90 days.
3. Missing values: 0 in all 17 columns.
4. Value ranges: humidity 27.35 to 100, temperature 12.06 to 30.88, wind 0.97 to 3.85, precipitation 0 to 42.5;
   holiday and activity flags only 0 or 1; no negative sales amounts. Discount ranges from 0 to 1.088:
   16,039 values are 0 and 34 are above 1. Decision: kept and flagged (disc_odd), because the stock-out
   metrics do not use discount.
5. Structure: each product belongs to 1 third-level category and each store to 1 city (maximum 1 in both checks).
6. Category hierarchy: each third-level category belongs to 1 second-level category and each second-level
   category to 1 first-level category (maximum 1 in both checks); 32 first-level categories.
7. Hourly check on all 4.5M rows: the stock-out count equals the hourly 1s in 100% of rows; statuses are only
   0 and 1; daily sales amounts equal the sum of hourly sales amounts (largest gap 4.6e-06, float32 rounding);
   0 negative hourly sales amounts.
8. sale_amount outliers: median 0.7, 99.9th percentile 20.6, maximum 44.9. 4,472 rows are above the 99.9th
   percentile. Decision: kept and flagged (sale_outlier), because the metrics use the stock-out columns,
   not sales.
9. Full-day stock-outs: 181,791 rows (4.0%) are out of stock in all 16 hours and hold 20.3% of all
   out-of-stock hours. Without them the hours share is 16.5% (not 19.9%) and the stock-out day rate 41.9%
   (not 44.3%), so the headline rates include these rows. Their sales amount is 0 in at least three quarters
   of them (mean 0.046, maximum 15.4). Possible explanation: the stock count covers hours 6 to 21 but daily
   sales cover 24 hours, so some sales may come from hours 0 to 5 or 22 to 23; to be checked in the analysis.
   Decision: kept and flagged (so_allday).
10. dt converted from text to dates; derived so_day (stock_hour6_22_cnt > 0) and so_share (count / 16).
11. ID columns downcast to small integers to save memory.
12. Reconciled: 4,500,000 rows, 50,000 series, 90 days each, stock-out hours match the hourly total.
    Stock-out day rate 0.4427, hours share 0.1988.
13. Saved as parquet in Project2_Stockout (19 MB). Not uploaded to GitHub: it is a derived file that the
    notebook rebuilds from the Hugging Face source.
14. The eval split (350,000 rows) is not used. The hourly lists stay in the source dataset for the
    hour-of-day work and were checked in batches.

- All-day stock-outs hold 20.3% of stock-out hours; the headline rates include them
  (without them, hours share 16.5% and day rate 41.9%).
- sale_amount is a normalised sales amount (units not stated).
- One retailer in China, one 90-day window, perishable goods only, so it does not describe Kenya.